# 04_analytics_gold — Transform Silver → Gold (Parquet)

This notebook reads Silver Delta, joins Postgres reference data (users), and writes analytics-ready Gold tables as Parquet.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from pyspark.sql import SparkSession


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(12):
        if (current / "docker-compose.yml").exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    raise RuntimeError("Could not find repo root (docker-compose.yml)")


repo_root = find_repo_root(Path.cwd())
lake_root = repo_root / "data_lake"

silver_path = str(lake_root / "silver" / "xlm_transactions")
gold_root = lake_root / "gold"
gold_hourly_path = str(gold_root / "xlm_hourly")
gold_daily_path = str(gold_root / "xlm_daily")
gold_country_daily_path = str(gold_root / "xlm_by_country_daily")

spark_master = os.environ.get("SPARK_MASTER", "local[*]")

spark = (
    SparkSession.builder.appName("engineering-project-xlm-gold")
    .master(spark_master)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config(
        "spark.jars.packages",
        ",".join(
            [
                "io.delta:delta-spark_2.12:3.2.0",
                "org.postgresql:postgresql:42.7.3",
            ]
        ),
    )
    .getOrCreate()
)

spark.version

In [ ]:
from pyspark.sql import functions as F

silver_df = spark.read.format("delta").load(silver_path)
silver_df.printSchema()
silver_df.show(5, truncate=False)

In [ ]:
jdbc_url = "jdbc:postgresql://postgres:5432/xlm"
jdbc_props = {
    "user": "xlm",
    "password": "xlm",
    "driver": "org.postgresql.Driver",
}

users_df = spark.read.jdbc(url=jdbc_url, table="users", properties=jdbc_props)
users_df.show(10, truncate=False)

In [ ]:
hourly_df = (
    silver_df.groupBy(F.date_trunc("hour", F.col("event_ts")).alias("hour_ts"))
    .agg(
        F.avg("price_usd").alias("avg_price_usd"),
        F.sum("volume_xlm").alias("total_volume_xlm"),
        F.sum("notional_usd").alias("total_notional_usd"),
        F.count(F.lit(1)).alias("tx_count"),
    )
)

daily_df = (
    silver_df.groupBy(F.date_trunc("day", F.col("event_ts")).alias("day_ts"))
    .agg(
        F.avg("price_usd").alias("avg_price_usd"),
        F.sum("volume_xlm").alias("total_volume_xlm"),
        F.sum("notional_usd").alias("total_notional_usd"),
        F.count(F.lit(1)).alias("tx_count"),
    )
)

by_country_daily_df = (
    silver_df.alias("s")
    .join(users_df.alias("u"), F.col("s.user_id") == F.col("u.user_id"), "inner")
    .groupBy(
        F.date_trunc("day", F.col("s.event_ts")).alias("day_ts"),
        F.col("u.country_code"),
    )
    .agg(
        F.avg(F.col("s.price_usd")).alias("avg_price_usd"),
        F.sum(F.col("s.volume_xlm")).alias("total_volume_xlm"),
        F.sum(F.col("s.notional_usd")).alias("total_notional_usd"),
        F.count(F.lit(1)).alias("tx_count"),
    )
)

hourly_df.show(5, truncate=False)
daily_df.show(5, truncate=False)
by_country_daily_df.show(5, truncate=False)

In [ ]:
gold_root.mkdir(parents=True, exist_ok=True)

hourly_df.write.mode("overwrite").parquet(gold_hourly_path)
daily_df.write.mode("overwrite").parquet(gold_daily_path)
by_country_daily_df.write.mode("overwrite").parquet(gold_country_daily_path)

gold_hourly_path, gold_daily_path, gold_country_daily_path